# Export fold-specific MRI embeddings
Uses saved checkpoints; no model training is performed.

For each outer fold, it:

1. loads the saved `best_validation_auc_checkpoint.pt`;
2. reconstructs only the MRI CNN–transformer encoder;
3. copies the trained MRI-encoder weights from the full checkpoint;
4. runs the existing preprocessed MRI volumes in evaluation mode under `torch.inference_mode()`;
5. saves the **masked 64-dimensional MRI representation** for all 544 MCI participants.

Participants without MRI are assigned an all-zero 64-D representation, matching the branch masking used by the multimodal model.

The five output files are written exactly where `standard_ml_baselines_mri_embeddings_thorough.ipynb` expects them.

**GPU is recommended for the one-time extraction step, but no training occurs.** The notebook can run on CPU as well, with a smaller batch size and slower inference.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from __future__ import annotations

import gc
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Fixed extraction configuration

ADNI_ROOT = Path("/content/drive/MyDrive/adni_mri")
MODEL_ROOT = ADNI_ROOT / "models" / "3mt_tmc_evidential"

TASK = "mci_prognosis"
EXPERIMENT_NAME = "gated_cmt_learned_gate_md050_seed_sensitivity"
SEED = 17
FOLDS = [0, 1, 2, 3, 4]

INPUT_ROOT = MODEL_ROOT / "final_task_ready_inputs" / TASK

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
    / TASK
    / f"seed_{SEED}"
)

OUTPUT_ROOT = (
    MODEL_ROOT
    / "standard_ml_baselines"
    / "mri_embeddings"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RID_COL = "RID"
TARGET_COL = "MCI_PROGNOSIS_TARGET"
ROLE_COL = "DATA_ROLE"
OUTER_FOLD_COL = "OUTER_FOLD"
MRI_PATH_COL = "MRI__NORMALIZED_T1_NPY_PATH"
MRI_MASK_COL = "BRANCH_MASK__MRI"

MRI_SHAPE = (177, 213, 183)
EMBEDDING_DIM = 64
EMBEDDING_COLS = [f"MRI_EMB_{i:02d}" for i in range(EMBEDDING_DIM)]

EXPECTED_PARTICIPANTS = 544

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 2 if DEVICE.type == "cuda" else 1
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

RESUME_EXISTING = True

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available: extraction will run on CPU and will be slower.")

print("Batch size:", BATCH_SIZE)
print("Output root:", OUTPUT_ROOT)


## Input checks

Checks that all five fold tables and all five **seed-17 best checkpoints** exist.


In [ ]:
def fold_input_path(fold: int) -> Path:
    return (
        INPUT_ROOT
        / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"
    )

def fold_checkpoint_path(fold: int) -> Path:
    return (
        EXPERIMENT_ROOT
        / f"fold_{fold}"
        / "checkpoints"
        / "best_validation_auc_checkpoint.pt"
    )

def fold_output_path(fold: int) -> Path:
    return (
        OUTPUT_ROOT
        / f"mci_prognosis_fold_{fold}_seed_{SEED}_mri_embeddings.csv"
    )

missing = []

for fold in FOLDS:
    input_path = fold_input_path(fold)
    checkpoint_path = fold_checkpoint_path(fold)

    if not input_path.is_file():
        missing.append(str(input_path))
    if not checkpoint_path.is_file():
        missing.append(str(checkpoint_path))

    print(f"Fold {fold}")
    print("  input:     ", input_path, "✓" if input_path.is_file() else "MISSING")
    print("  checkpoint:", checkpoint_path, "✓" if checkpoint_path.is_file() else "MISSING")
    print("  output:    ", fold_output_path(fold))

if missing:
    raise FileNotFoundError(
        "Required files are missing. No extraction has started.\n\n"
        + "\n".join(missing)
    )

print("\nInput check passed for all five folds.")


## Exact MRI encoder architecture

The two classes below reproduce the MRI branch used in the trained model:

`177 × 213 × 183 MRI → 3D CNN stem → four residual downsampling blocks → patch-wise transformer → mean pooling → 64-D projection`

Only this trained submodule is reconstructed. The rest of the 3MT/TMC model is unnecessary for feature extraction.


In [ ]:
class ResidualDownsampleBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()

    def forward(self, inputs):
        pooled_inputs = self.pool(inputs)
        main_features = self.main_path(pooled_inputs)
        residual_features = self.residual_path(pooled_inputs)

        return self.activation(
            main_features + residual_features
        )

class MRIEncoder3D(nn.Module):
    def __init__(
        self,
        output_dim,
        input_shape=MRI_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be divisible "
                "by the number of attention heads."
            )

        self.input_shape = tuple(input_shape)
        self.patch_embedding_dim = patch_embedding_dim

        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(16, affine=True),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(16, affine=True),
            nn.GELU(),
        )

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(16, 32),
            ResidualDownsampleBlock3D(32, 64),
            ResidualDownsampleBlock3D(64, 128),
            ResidualDownsampleBlock3D(128, 256),
        )

        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        patch_grid_shape = stem_shape
        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(dimension < 1 for dimension in patch_grid_shape):
            raise ValueError(
                "The MRI input becomes too small after convolutional downsampling."
            )

        self.patch_grid_shape = patch_grid_shape
        self.number_of_patches = math.prod(patch_grid_shape)

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(patch_embedding_dim),
        )

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, image):
        feature_map = self.stem(image)
        feature_map = self.residual_blocks(feature_map)

        batch_size, channels, depth, height, width = feature_map.shape

        actual_patch_count = depth * height * width
        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        patch_tokens = patch_tokens + self.position_embedding

        transformed_tokens = self.transformer_encoder(
            patch_tokens
        )

        pooled_representation = transformed_tokens.mean(dim=1)

        return self.projection(
            pooled_representation
        )

test_encoder = MRIEncoder3D(output_dim=EMBEDDING_DIM)
print("MRI patch grid:", test_encoder.patch_grid_shape)
print("MRI patch count:", test_encoder.number_of_patches)
print("Output dimension:", EMBEDDING_DIM)
del test_encoder


## Load only the trained MRI weights

The checkpoint contains the complete multimodal model state. This helper identifies the MRI submodule inside that state dictionary and copies **only** its trained parameters into the standalone MRI encoder.

It also checks that the checkpoint identity matches the requested fold, seed, task and experiment.


In [ ]:
def torch_load_checkpoint(path: Path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        # Compatibility with older PyTorch versions.
        return torch.load(
            path,
            map_location="cpu",
        )

def validate_checkpoint_identity(checkpoint, fold: int, path: Path):
    checks = {
        "fold": (checkpoint.get("fold"), fold),
        "random_seed": (checkpoint.get("random_seed"), SEED),
        "task": (checkpoint.get("task"), TASK),
        "experiment_name": (
            checkpoint.get("experiment_name"),
            EXPERIMENT_NAME,
        ),
    }

    problems = []
    for name, (actual, expected) in checks.items():
        if actual != expected:
            problems.append(
                f"{name}: checkpoint={actual!r}, expected={expected!r}"
            )

    if problems:
        raise RuntimeError(
            f"Checkpoint identity mismatch for {path}:\n"
            + "\n".join(problems)
        )

def find_mri_state_prefix(full_state_dict, standalone_encoder):
    expected_keys = list(
        standalone_encoder.state_dict().keys()
    )

    # position_embedding is distinctive and gives a cheap first search.
    anchor = "position_embedding"

    candidate_prefixes = []
    for full_key in full_state_dict.keys():
        if full_key.endswith(anchor):
            prefix = full_key[:-len(anchor)]
            if all(
                prefix + key in full_state_dict
                for key in expected_keys
            ):
                candidate_prefixes.append(prefix)

    # fallback in case a future naming variant changes the anchor.
    if not candidate_prefixes:
        possible_prefixes = set()

        for full_key in full_state_dict.keys():
            for expected_key in expected_keys:
                if full_key.endswith(expected_key):
                    possible_prefixes.add(
                        full_key[:-len(expected_key)]
                    )

        candidate_prefixes = [
            prefix
            for prefix in possible_prefixes
            if all(
                prefix + key in full_state_dict
                for key in expected_keys
            )
        ]

    candidate_prefixes = sorted(set(candidate_prefixes))

    if len(candidate_prefixes) != 1:
        raise RuntimeError(
            "Could not identify exactly one MRI-encoder prefix in the "
            "checkpoint state_dict. Candidates:\n"
            + "\n".join(candidate_prefixes)
        )

    return candidate_prefixes[0]

def load_trained_mri_encoder(fold: int):
    checkpoint_path = fold_checkpoint_path(fold)

    checkpoint = torch_load_checkpoint(
        checkpoint_path
    )
    validate_checkpoint_identity(
        checkpoint=checkpoint,
        fold=fold,
        path=checkpoint_path,
    )

    if "model_state_dict" not in checkpoint:
        raise KeyError(
            f"{checkpoint_path} has no model_state_dict."
        )

    encoder = MRIEncoder3D(
        output_dim=EMBEDDING_DIM
    )

    full_state = checkpoint["model_state_dict"]
    prefix = find_mri_state_prefix(
        full_state_dict=full_state,
        standalone_encoder=encoder,
    )

    expected_keys = encoder.state_dict().keys()

    mri_state = {
        key: full_state[prefix + key]
        for key in expected_keys
    }

    encoder.load_state_dict(
        mri_state,
        strict=True,
    )

    checkpoint_epoch = int(
        checkpoint.get("epoch", -1)
    )
    best_validation_auc = float(
        checkpoint.get(
            "best_validation_auc",
            np.nan,
        )
    )

    del mri_state
    del full_state
    del checkpoint
    gc.collect()

    encoder = encoder.to(DEVICE)
    encoder.eval()

    print(
        f"Fold {fold}: loaded MRI weights from prefix "
        f"{prefix!r}"
    )
    print(
        f"Fold {fold}: checkpoint epoch={checkpoint_epoch}, "
        f"best validation AUC={best_validation_auc:.6f}"
    )

    return (
        encoder,
        checkpoint_epoch,
        best_validation_auc,
        prefix,
    )


## MRI loading and output validation

Only participants with `BRANCH_MASK__MRI == 1` are read from disk and passed through the encoder.

For participants without MRI, the saved embedding remains exactly zero. This reproduces the **masked MRI representation** that enters the original multimodal fusion pathways.


In [ ]:
class AvailableMRIDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame):
        self.df = (
            dataframe
            .reset_index()
            .rename(columns={"index": "_original_row"})
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        rid = int(row[RID_COL])
        original_row = int(row["_original_row"])

        path_value = row[MRI_PATH_COL]

        if pd.isna(path_value):
            raise ValueError(
                f"RID {rid}: MRI is marked available but MRI path is missing."
            )

        mri_path = Path(str(path_value))

        if not mri_path.is_file():
            raise FileNotFoundError(
                f"RID {rid}: MRI file not found:\n{mri_path}"
            )

        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SHAPE:
            raise ValueError(
                f"RID {rid}: unexpected MRI shape {mri_array.shape}; "
                f"expected {MRI_SHAPE}.\n{mri_path}"
            )

        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        if not np.isfinite(mri_array).all():
            raise ValueError(
                f"RID {rid}: MRI contains NaN or infinite values."
            )

        image = torch.from_numpy(
            mri_array[None, ...]
        )

        return {
            "original_row": original_row,
            "rid": rid,
            "image": image,
        }

def validate_fold_table(df: pd.DataFrame, fold: int):
    required = {
        RID_COL,
        TARGET_COL,
        ROLE_COL,
        OUTER_FOLD_COL,
        MRI_PATH_COL,
        MRI_MASK_COL,
    }

    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(
            f"Fold {fold}: input table is missing columns:\n{missing}"
        )

    if len(df) != EXPECTED_PARTICIPANTS:
        raise ValueError(
            f"Fold {fold}: expected {EXPECTED_PARTICIPANTS} rows, "
            f"found {len(df)}."
        )

    rid = pd.to_numeric(
        df[RID_COL],
        errors="raise",
    ).astype(int)

    if rid.duplicated().any():
        raise ValueError(
            f"Fold {fold}: duplicate RID values."
        )

    mask = pd.to_numeric(
        df[MRI_MASK_COL],
        errors="raise",
    )

    if not mask.isin([0, 1]).all():
        raise ValueError(
            f"Fold {fold}: {MRI_MASK_COL} must contain only 0/1."
        )

    test_folds = (
        pd.to_numeric(
            df.loc[df[ROLE_COL].astype(str).str.lower() == "test", OUTER_FOLD_COL],
            errors="raise",
        )
        .astype(int)
        .unique()
    )

    if len(test_folds) != 1 or int(test_folds[0]) != fold:
        raise ValueError(
            f"Fold {fold}: test rows do not identify the expected outer fold. "
            f"Found {test_folds.tolist()}."
        )

    return mask.astype(int)

def validate_saved_embedding_table(
    output: pd.DataFrame,
    source_df: pd.DataFrame,
    fold: int,
):
    required = {
        RID_COL,
        TARGET_COL,
        ROLE_COL,
        MRI_MASK_COL,
        *EMBEDDING_COLS,
    }

    missing = sorted(required - set(output.columns))
    if missing:
        raise ValueError(
            f"Fold {fold}: output is missing columns:\n{missing}"
        )

    if len(output) != EXPECTED_PARTICIPANTS:
        raise ValueError(
            f"Fold {fold}: output has {len(output)} rows; "
            f"expected {EXPECTED_PARTICIPANTS}."
        )

    if output[RID_COL].duplicated().any():
        raise ValueError(
            f"Fold {fold}: output contains duplicate RIDs."
        )

    if set(output[RID_COL].astype(int)) != set(source_df[RID_COL].astype(int)):
        raise ValueError(
            f"Fold {fold}: output RID set does not match source fold table."
        )

    values = output[EMBEDDING_COLS].to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(values).all():
        raise ValueError(
            f"Fold {fold}: output embeddings contain NaN/inf."
        )

    masks = output[MRI_MASK_COL].to_numpy(
        dtype=int
    )

    unavailable = masks == 0
    if np.any(unavailable):
        if not np.allclose(
            values[unavailable],
            0.0,
            atol=1e-8,
        ):
            raise ValueError(
                f"Fold {fold}: unavailable MRI rows are not exactly zero."
            )

    available = masks == 1
    if np.any(available):
        if np.allclose(
            values[available],
            0.0,
            atol=1e-12,
        ):
            raise ValueError(
                f"Fold {fold}: all available-MRI embeddings are zero."
            )

    source_masks = (
        source_df[[RID_COL, MRI_MASK_COL]]
        .copy()
    )
    check = source_masks.merge(
        output[[RID_COL, MRI_MASK_COL]],
        on=RID_COL,
        how="inner",
        validate="one_to_one",
        suffixes=("_source", "_output"),
    )

    if not (
        check[f"{MRI_MASK_COL}_source"].astype(int)
        == check[f"{MRI_MASK_COL}_output"].astype(int)
    ).all():
        raise ValueError(
            f"Fold {fold}: MRI masks changed during export."
        )

def existing_output_is_valid(path: Path, source_df: pd.DataFrame, fold: int):
    if not path.is_file():
        return False

    try:
        existing = pd.read_csv(path)
        validate_saved_embedding_table(
            output=existing,
            source_df=source_df,
            fold=fold,
        )
        return True
    except Exception as exc:
        print(
            f"Fold {fold}: existing output is invalid and will be replaced:\n"
            f"{exc}"
        )
        return False


## Extract one fold

Inference only. The encoder is in `.eval()` mode and the forward passes run inside `torch.inference_mode()`.

The function writes the CSV atomically only after the entire fold passes validation.


In [ ]:
def extract_fold_embeddings(
    fold: int,
    batch_size: int = BATCH_SIZE,
):
    input_path = fold_input_path(fold)
    output_path = fold_output_path(fold)

    df = pd.read_csv(input_path)
    mri_mask = validate_fold_table(
        df=df,
        fold=fold,
    )

    df = df.copy()
    df[RID_COL] = pd.to_numeric(
        df[RID_COL],
        errors="raise",
    ).astype(int)

    df[MRI_MASK_COL] = mri_mask.astype(int)

    if (
        RESUME_EXISTING
        and existing_output_is_valid(
            path=output_path,
            source_df=df,
            fold=fold,
        )
    ):
        print(
            f"Fold {fold}: valid embedding file already exists; skipping."
        )
        return pd.read_csv(output_path)

    available_df = df.loc[
        df[MRI_MASK_COL] == 1
    ].copy()

    print("\n" + "=" * 72)
    print(f"FOLD {fold}")
    print("=" * 72)
    print("Participants:", len(df))
    print("MRI available:", len(available_df))
    print("MRI unavailable:", len(df) - len(available_df))

    (
        encoder,
        checkpoint_epoch,
        best_validation_auc,
        state_prefix,
    ) = load_trained_mri_encoder(fold)

    embeddings = np.zeros(
        (len(df), EMBEDDING_DIM),
        dtype=np.float32,
    )

    dataset = AvailableMRIDataset(
        available_df
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    try:
        with torch.inference_mode():
            for batch in tqdm(
                loader,
                desc=f"Fold {fold} MRI embeddings",
            ):
                images = batch["image"].to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                batch_embeddings = encoder(
                    images
                )

                if batch_embeddings.ndim != 2:
                    raise RuntimeError(
                        f"Fold {fold}: encoder returned shape "
                        f"{tuple(batch_embeddings.shape)}"
                    )

                if batch_embeddings.shape[1] != EMBEDDING_DIM:
                    raise RuntimeError(
                        f"Fold {fold}: encoder returned "
                        f"{batch_embeddings.shape[1]} dimensions; "
                        f"expected {EMBEDDING_DIM}."
                    )

                batch_embeddings = (
                    batch_embeddings
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32, copy=False)
                )

                original_rows = (
                    batch["original_row"]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(int)
                )

                embeddings[
                    original_rows,
                    :
                ] = batch_embeddings

                del images
                del batch_embeddings

    except RuntimeError as exc:
        if (
            DEVICE.type == "cuda"
            and "out of memory" in str(exc).lower()
            and batch_size > 1
        ):
            print(
                f"\nFold {fold}: CUDA out of memory with batch size "
                f"{batch_size}. Retrying this fold with batch size 1."
            )

            del loader
            del dataset
            del encoder
            gc.collect()
            torch.cuda.empty_cache()

            return extract_fold_embeddings(
                fold=fold,
                batch_size=1,
            )

        raise

    if not np.isfinite(embeddings).all():
        raise ValueError(
            f"Fold {fold}: extracted matrix contains NaN/inf."
        )

    output = pd.DataFrame({
        RID_COL: df[RID_COL].to_numpy(dtype=int),
        TARGET_COL: pd.to_numeric(
            df[TARGET_COL],
            errors="raise",
        ).to_numpy(dtype=int),
        OUTER_FOLD_COL: df[OUTER_FOLD_COL].to_numpy(),
        ROLE_COL: df[ROLE_COL].astype(str).to_numpy(),
        MRI_MASK_COL: df[MRI_MASK_COL].to_numpy(dtype=int),
        "MRI_EMBEDDING_SEED": SEED,
        "MRI_ENCODER_CHECKPOINT_EPOCH": checkpoint_epoch,
        "MRI_ENCODER_BEST_VALIDATION_AUC": best_validation_auc,
        "MRI_ENCODER_STATE_PREFIX": state_prefix,
    })

    embedding_frame = pd.DataFrame(
        embeddings,
        columns=EMBEDDING_COLS,
    )

    output = pd.concat(
        [
            output.reset_index(drop=True),
            embedding_frame.reset_index(drop=True),
        ],
        axis=1,
    )

    validate_saved_embedding_table(
        output=output,
        source_df=df,
        fold=fold,
    )

    temporary_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    output.to_csv(
        temporary_path,
        index=False,
    )

    os.replace(
        temporary_path,
        output_path,
    )

    print(
        f"\nFold {fold}: saved {len(output)} rows × "
        f"{EMBEDDING_DIM} MRI dimensions."
    )
    print(output_path)

    del loader
    del dataset
    del encoder
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return output


## Run all five folds

 Completed, valid fold files are skipped automatically, so if the Colab session stops after some folds the run can rerun the notebook and continue.


In [ ]:
all_summaries = []

for fold in FOLDS:
    output = extract_fold_embeddings(
        fold=fold
    )

    values = output[EMBEDDING_COLS].to_numpy(
        dtype=np.float64
    )
    masks = output[MRI_MASK_COL].to_numpy(
        dtype=int
    )

    available_values = values[
        masks == 1
    ]

    all_summaries.append({
        "fold": fold,
        "seed": SEED,
        "rows": len(output),
        "mri_available": int((masks == 1).sum()),
        "mri_unavailable": int((masks == 0).sum()),
        "embedding_dim": len(EMBEDDING_COLS),
        "available_embedding_abs_mean": (
            float(np.abs(available_values).mean())
            if len(available_values)
            else np.nan
        ),
        "output_path": str(fold_output_path(fold)),
    })

summary = pd.DataFrame(
    all_summaries
)

display(summary)


## Output check

This verifies the five filenames expected by the standard-ML notebook.


In [ ]:
expected_outputs = [
    fold_output_path(fold)
    for fold in FOLDS
]

print("=" * 72)
print("MRI EMBEDDING EXPORT COMPLETE")
print("=" * 72)

for path in expected_outputs:
    status = "✓" if path.is_file() else "MISSING"
    print(status, path.name)

if not all(path.is_file() for path in expected_outputs):
    raise FileNotFoundError(
        "At least one expected MRI embedding CSV is still missing."
    )

print(
    "\nAll five seed-17 fold-specific MRI embedding files exist.\n"
    "Next: run standard_ml_baselines_mri_embeddings_thorough.ipynb."
)
